# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv(dotenv_path='../../.env')
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

   

In [17]:
result = con.sql("""
    SELECT COUNT(DISTINCT content_hash_id) AS distinct_pages, COUNT(*) AS total_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result)

   distinct_pages  total_rows
0          331437     9841378


In the table, one row means, the performance of one page on a one specific day. So if a page existed for every day of march, it would show up 31 times(once every day).

I'm using data from March 2026 (month=2026-03),that's a safe middle month, 
not the very last month, since the last month is meant to be a sealed 
test later, so it cannot be used to develop my own logic.

I checked this with two real queries:
- The whole month has 9,841,378 rows, dated March 1st to March 31st.
- There are 331,437 distinct pages in this data.
- On average, each page appears about 29.7 times (9,841,378 / 331,437), 
  which is close to but not exactly 31. This makes sense, not every 
  page existed for the entire month, some may have started in between
  or may have missing days.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Before sorting columns into boxes, I first checked what columns exist in this table, instead of guessing. I used a DESCRIBE query to list every column name and its type.

In [18]:
result = con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

I sorted the columns into 4 boxes for my CTR/position question:

Feature (real numbers that I'd know befor deciding and are useful for ctr): 
- gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available

Label/proxy (what my CTR gap target comes from): 
- gsc_clicks and gsc_impressions together, since CTR = clicks / impressions

Context (just ID tags, not numbers to calculate with): 
- client_hash_id, content_hash_id, report_date, month

Excluded (real columns, but not used with reasons):
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, 
  ai_other -- these are about AI referral traffic, not search CTR, so 
  they're off-topic for my question
- ga4_pageviews, ga4_sessions, ga4_users, and other ga4_ columns -- 
  also off-topic (not about search position/CTR), and risky to use 
  without checking ga4_data_available first, since some rows are 
  zero-filled fake data, not real numbers

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
result = con.sql("""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print(result)
print(f"Rows returned: {len(result)} -- if 0, the grain holds (one row per page-client-day).")

Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, c]
Index: []
Rows returned: 0 -- if 0, the grain holds (one row per page-client-day).


In [20]:
result = con.sql("""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc_data
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result)

   total_rows  rows_with_gsc_data
0     9841378           3611061.0


I ran 3 checks to prove my contract claims:

1. Grain check: I grouped by page, client, date, looking for any group with more than 1 row. It returned 0 rows that confirms one row really is one page-client-day, with no duplicates.

2. Row count + date span (from section 1): March 2026 has 9,841,378 rows, dated March 1st to March 31st.

3. Availability check: out of 9,841,378 total rows, only 3,611,061 (about 36.7%) actually have real GSC data available (gsc_data_available IS TRUE). This means most rows in this month don't have usable search data yet, an important thing to filter for before building features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [21]:
features = con.sql("""
    SELECT 
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 100.0 / gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    LIMIT 500000
""").df()

print(features.shape)
features.head()

(500000, 7)


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,0.0
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,0.0
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,0.8
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,0.0
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,0.0


My 5 features for this lane are:

1. gsc_impressions - knowable at the decision moment because before I'd make any recommendation, it's already recorded for the that day.

2. gsc_clicks - knowable at the decision moment because, like impressions, it's already recorded for that day.

3. gsc_avg_position - knowable at the decision moment because it reflects where the page already ranked that day, which is a real observed fact, not a future prediction.

4. ctr (calculated from clicks/impressions) - knowable at the decision moment because it's built from pieces(gsc_clicks, gsc_impressions) already known for that day.

5. report_date - knowable at the decision moment because it's just the date the row is for, always known.

In [22]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# build the target: CTR gap = how far below the day's average CTR a page falls
features['day_avg_ctr'] = features.groupby('report_date')['ctr'].transform('mean')
features['ctr_gap'] = features['day_avg_ctr'] - features['ctr']

# HONEST features only
X_honest = features[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']].fillna(0)
y = features['ctr_gap'].fillna(0)

model = LinearRegression()
model.fit(X_honest, y)
pred = model.predict(X_honest)
honest_score = r2_score(y, pred)

print(f"Honest R² score (using only real features): {honest_score:.4f}")

Honest R² score (using only real features): 0.0158


In [ ]:
#cheating on purpose
features['cheat_column'] = features['ctr_gap'] * 0.99  # basically the answer

X_cheat = features[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'cheat_column']].fillna(0)

model_cheat = LinearRegression()
model_cheat.fit(X_cheat, y)
pred_cheat = model_cheat.predict(X_cheat)
cheat_score = r2_score(y, pred_cheat)

print(f"Honest R² score: {honest_score:.4f}")
print(f"CHEATING R² score (with leaked column): {cheat_score:.4f}")
print(f"Jump: {cheat_score - honest_score:.4f}")

Honest R² score: 0.0158
CHEATING R² score (with leaked column): 1.0000
Jump: 0.9842


In [24]:
features = features.drop(columns=['cheat_column'])
print("cheat column removed, honest score stays:", honest_score)

cheat column removed, honest score stays: 0.015815618859730707


I tested what will happen if I sneak in a column that is  built from the answer itself. My honest score was only 0.0158 But when I added a "cheat" column that is made from the actual target, the score jumped to a perfect 1.0000. This jump of 0.9842 is a big red flag, a real score should never look that perfect. I deleted the cheat column and kept my honest score of 0.0158.

LIMITATION OF MY FINDING: 

 One limitation of my slice is that, only about 36.7% of March's rows actually have real GSC data available (gsc_data_available IS TRUE). The rest of the data is not usable for search-position analysis. This means my results only reflect pages/clients with enough search history, clients or pages with shorter or missing history are basically invisible in this slice, so my findings might not apply evenly to all of the FlyRank's clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.